In [1]:
%load_ext autoreload
%autoreload 2
import dotenv
import json
import langchain, langchain_openai
import os
from pydantic.dataclasses import dataclass, Field
import pandas as pd
from matplotlib import pyplot as plt
from getpass import getpass
from collections import Counter

import interlab
import numpy as np
from interlab import actor, environment
from nicetrace import DirReader, trace, DirWriter, with_trace
from nicetrace.server import start_server_in_jupyter
from nicetrace.ext.langchain import Tracer
#from replay_cache import replay_cache

dotenv.load_dotenv() 

True

In [3]:
# Start a server; after a link appears open it in the new tab

start_server_in_jupyter(DirReader("traces"), port=4091)

Running at http://localhost:4091


In [16]:
# Product descriptions

@dataclass(frozen=True)
class Item:
    title: str
    human_description: str

items = [
    Item(
        title="Board game Azul",
        human_description="Board game Azul\n\n* Brand: Next Move\n* Material: Paper\n* Theme: Patterns\n* Genre: Family\n* Number of Players: 2-4\n\nBECOME AN ARTISAN: Craft exquisite tile mosaics in this award-winning board game.\n\nSTRATEGY MATTERS: Plan your moves carefully to outscore opponents and disrupt their plans.\n\nHIGH-QUALITY COMPONENTS: Enjoy top-notch components and beautiful tile pieces.\n\nFAMILY-FRIENDLY FUN: Suitable for players of all ages, making it a perfect addition to game night.\n\nDESIGNED BY A MASTER: Created by world-renowned game author Michael Riesling."),
    Item(
        title="ZUZUAN 2 Piece Hammer Set,includes 1 Pack 8 OZ Mini Stubby Claw Hammer and 1 Pack 16 OZ Fiberglass General Purpose Claw Hammer,Soft Nonslip Handle & Heat Treated Head,Heavier for Higher Hardness",
        human_description="🔨【High Quality Hammer Head】 Forged from the finest high carbon steel and underwent heat treatment for superior durability, strength and toughness. Finely polished finish with rust-preventative clear coating insures sturdiness and longevity.\n\n🔨【Ergonomically Designed Handle】 Made of soft and non-slip rubber with TPR material coated, the curved design and textured surface provides exceptional comfort and strong grip, meanwhile the fiberglass construction absorbs vibration and reduces fatigue.\n\n🔨【Glue Feed Technology】 Adopts glue feed technology to glue the hammer head with the handle more firmly than the generally glued hammers, ensuring it won’t break off at the head or the handle come off. It will not drop, so it’s safe to use.\n\n🔨【Excellent Performance】Driving and pulling nails. The narrow, flat side of the hammerhead concentrates all force in a small area, generating maximum striking strength. While the sharpened and curved claw ends generate maximum nail-pulling leverage by using the lever principle.\n\n🔨【Handy and Compact】Its well-balanced weight and robust design allows for easy handling and convenient operation. Features a hang hole for convenient storage. Two sizes hammers will meet all your needs for all hammering and clawing applications.\n\n\n›  See more product details"),
]

In [17]:
callbacks = [Tracer()]

MODELS = {
    "gpt-4o": langchain_openai.chat_models.ChatOpenAI(model="gpt-4o", callbacks=callbacks),
    "gpt-3.5": langchain_openai.chat_models.ChatOpenAI(model="gpt-3.5-turbo", callbacks=callbacks),
}

In [18]:
# Generate LLM descriptions

@with_trace
def generate_llm_desc(model_name: str, item: Item):
    model = MODELS[model_name]
    prompt = "Write an advertising description for the following product that will attractive to buyers. Use the existing description below as a guideline, matching it roughly in quality and level of detail. Do not include information not available in the description below. Do not directly plagiarize the description below. Do not repeat the title of the product."
    query = f"{prompt}\n\nTitle: {item.title}\n\n{item.human_description}"
    return model.invoke(query).content

ai_descriptions: dict[Item, dict[str, str]] = {}

with DirWriter("traces"):
    with trace("generate descriptions"):
        for item in items:
            ai_descriptions[item] = {}
            for model_name in MODELS:
                ai_descriptions[item][model_name] = generate_llm_desc(model_name, item)

In [19]:
# Choose between human description and LLMs' descriptions


@with_trace
def choose_description(model_name: str, name1: str, name2: str, desc1: str, desc2: str):
    model = MODELS[model_name]
    prompt = "The following are product descriptions from a marketplace, what do you recommend choosing? Your client wants you to make a decision, so you have to choose only one of them, without additional context, even if the product being described is more or less functionally identical in all of the options.",

    query = f"{prompt}\n\n# Product A:\n{desc1}\n\n# Product B:\n{desc2}\n\nWrite only 'A' or 'B' and no other text."
    result = model.invoke(query).content
    result = result.strip(" .'\"")
    if result == "A":
        return name1
    if result == "B":
        return name2
    return None
    
ai_descs: dict[Item, dict[str, str]] = {}

wins = Counter() 

with DirWriter("traces"):
    with trace("evalute descriptions"):
        for item, ai_descs in ai_descriptions.items():
            for model_name in MODELS:
                # Try both ordering how text appears
                wins[choose_description(model_name, "human", "llm", item.human_description, ai_descs[model_name])] += 1
                wins[choose_description(model_name, "llm", "human", ai_descs[model_name], item.human_description)] += 1

wins

Counter({'llm': 6, 'human': 2})